## Anomaly Clustering Evaluation
* Evaluate clustering through:
    * Silhouette Score
    * Davies-Bouldin Index and Calinski-Harabasz Index
    * Internal Cluster Density and Separation Metrics
    * Connectivity or Nearest-Neighbor Analysis

#### Imports

In [1]:
import pickle
import os
import pandas as pd
from ocean_tools.processing.clustering import extract_experiment_aggregated_features, compute_clustering_metrics
from ocean_tools.io.writers import store_pickle_variable

ExperimentSet = {
    'experiment_structure': ['variable_list', 'start_date', 'end_date', 'anomaly_threshold_list', 'eps_t', 'eps_lat', 'eps_lon', 'min_neighbors', 'min_cluster_size', 'experiment_name'],
    'experiments': [
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 40, 40, 300, 1, 'Base Full Period'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 40, 40, 100, 1, 'Smaller Min Neighbors'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 40, 40, 500, 1, 'Larger Min Neighbors'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 40, 40, 1000, 1, 'Much Larger Min Neighbors'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 20, 20, 300, 1, 'Smaller Geo Eps'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 60, 60, 300, 1, 'Larger Geo Eps'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 2, 40, 40, 300, 1, 'Larger Time Eps'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 12, 40, 40, 300, 1, 'Much Larger Time Eps'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 20, 20, 500, 1, '(2) Base Full Period'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 20, 20, 250, 1, '(2) Smaller Min Neighbors'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 20, 20, 750, 1, '(2) Larger Min Neighbors'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 20, 20, 1000, 1, '(2) Much Larger Min Neighbors'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 10, 10, 500, 1, '(2) Smaller Geo Eps'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 1, 50, 50, 500, 1, '(2) Larger Geo Eps'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 2, 20, 20, 500, 1, '(2) Larger Time Eps'],
        [['sst', 'chlor_a'], '2002-08-01', '2025-01-01', [2, 5], 12, 20, 20, 500, 1, '(2) Much Larger Time Eps'],
    ]
}

#### Summarized Cluster Features

In [2]:
# Read stored features
import pickle
import os

exports_path = './data/exports/clusters/mv_experiments/features/'
experiment_features_1 = pickle.load(open(os.path.join(exports_path, f"experiment_features_run_1.pkl"), 'rb'))
experiment_features_2 = pickle.load(open(os.path.join(exports_path, f"experiment_features_run_2.pkl"), 'rb'))

experiment_features = experiment_features_1 + experiment_features_2

print(f"Number of experiments loaded: {len(experiment_features)}")

# Compute aggregated metrics for each experiment.
experiment_level_features = [extract_experiment_aggregated_features(exp) for exp in experiment_features]

# Convert to a DataFrame.
df_experiment_features = pd.DataFrame(experiment_level_features)
print(df_experiment_features.to_string(index=False))

Number of experiments loaded: 16
              experiment_name                                                                      file_name  num_clusters  time_length_mean  time_length_std  time_length_min  time_length_max  lat_length_mean  lat_length_std  lat_length_min  lat_length_max  lon_length_mean  lon_length_std  lon_length_min  lon_length_max  cluster_size_mean  cluster_size_std  cluster_size_min  cluster_size_max  cluster_volume_mean  cluster_volume_std  cluster_volume_min  cluster_volume_max  cluster_compactness_mean  cluster_compactness_std  cluster_compactness_min  cluster_compactness_max  cluster_eccentricity_mean  cluster_eccentricity_std  cluster_eccentricity_min  cluster_eccentricity_max  cluster_dispersion_mean  cluster_dispersion_std  cluster_dispersion_min  cluster_dispersion_max
             Base Full Period  clustering_experiment_2002-08-01_2025-01-01_sst-chlor_a_2-5_1_40_40_300_1.pkl            15          2.533333         1.820867              1.0              

#### Evaluation Metrics From Clustered Data

In [7]:
# Read Stored Clusters.
exports_path = './data/exports/clusters/mv_experiments/'
experiment_results = []
for experiment in ExperimentSet['experiments']:
    variable_list, start_date, end_date, anomaly_threshold_list, eps_t, eps_lat, eps_lon, min_neighbors, min_cluster_size, experiment_name = experiment
    file_name = f"clustering_experiment_{start_date}_{end_date}_{'-'.join(variable_list)}_{'-'.join(map(str, anomaly_threshold_list))}_{eps_t}_{eps_lat}_{eps_lon}_{min_neighbors}_{min_cluster_size}.pkl"
    clusters, n_clusters, n_discarded, run_seconds = pickle.load(open(os.path.join(exports_path, file_name), 'rb'))
    experiment_results.append((clusters, n_clusters, n_discarded, run_seconds, file_name, experiment_name))

In [8]:
# Compute metrics for each round 1 experiment.
experiment_metrics = []
for experiment_result in experiment_results:
    clusters, n_clusters, n_discarded, run_seconds, file_name, experiment_name = experiment_result
    metrics = compute_clustering_metrics(clusters)
    experiment_metrics.append((experiment_name, metrics))
    print(f"Experiment: {experiment_name} | Silhouette: {metrics['silhouette']:.3f} | Davies-Bouldin: {metrics['davies_bouldin']:.3f} | Calinski-Harabasz: {metrics['calinski_harabasz']:.3f} | Noise Ratio: {metrics['noise_ratio']:.3f}")

Experiment: Base Full Period | Silhouette: 0.165 | Davies-Bouldin: 1.486 | Calinski-Harabasz: 9578.049 | Noise Ratio: 0.229
Experiment: Smaller Min Neighbors | Silhouette: -0.016 | Davies-Bouldin: 1.725 | Calinski-Harabasz: 3034.477 | Noise Ratio: 0.078
Experiment: Larger Min Neighbors | Silhouette: 0.291 | Davies-Bouldin: 1.042 | Calinski-Harabasz: 14101.078 | Noise Ratio: 0.377
Experiment: Much Larger Min Neighbors | Silhouette: 0.558 | Davies-Bouldin: 0.725 | Calinski-Harabasz: 20275.570 | Noise Ratio: 0.645
Experiment: Smaller Geo Eps | Silhouette: 0.332 | Davies-Bouldin: 1.017 | Calinski-Harabasz: 15600.534 | Noise Ratio: 0.419
Experiment: Larger Geo Eps | Silhouette: 0.131 | Davies-Bouldin: 1.933 | Calinski-Harabasz: 7251.014 | Noise Ratio: 0.177
Experiment: Larger Time Eps | Silhouette: 0.167 | Davies-Bouldin: 1.604 | Calinski-Harabasz: 6400.820 | Noise Ratio: 0.219
Experiment: Much Larger Time Eps | Silhouette: -0.064 | Davies-Bouldin: 1.263 | Calinski-Harabasz: 2637.494 | Nois

In [9]:
# Store all results.
exports_path = './data/exports/clusters/mv_experiments/evaluation_metrics/'
file_name = f"clustering_metrics_run_1"
store_pickle_variable(experiment_metrics, exports_path, file_name)

In [3]:
# Compute metrics for each round 2 experiment.
experiment_metrics = []
for experiment_result in experiment_results:
    clusters, n_clusters, n_discarded, run_seconds, file_name, experiment_name = experiment_result
    metrics = compute_clustering_metrics(clusters)
    experiment_metrics.append((experiment_name, metrics))
    print(f"Experiment: {experiment_name} | Silhouette: {metrics['silhouette']:.3f} | Davies-Bouldin: {metrics['davies_bouldin']:.3f} | Calinski-Harabasz: {metrics['calinski_harabasz']:.3f} | Noise Ratio: {metrics['noise_ratio']:.3f}")

Experiment: (2) Base Full Period | Silhouette: 0.420 | Davies-Bouldin: 0.796 | Calinski-Harabasz: 17593.532 | Noise Ratio: 0.610
Experiment: (2) Smaller Min Neighbors | Silhouette: 0.258 | Davies-Bouldin: 1.034 | Calinski-Harabasz: 12086.811 | Noise Ratio: 0.320
Experiment: (2) Larger Min Neighbors | Silhouette: 0.802 | Davies-Bouldin: 0.206 | Calinski-Harabasz: 6898.648 | Noise Ratio: 0.801
Experiment: (2) Much Larger Min Neighbors | Silhouette: nan | Davies-Bouldin: nan | Calinski-Harabasz: nan | Noise Ratio: 0.835
Experiment: (2) Smaller Geo Eps | Silhouette: nan | Davies-Bouldin: nan | Calinski-Harabasz: nan | Noise Ratio: 0.978
Experiment: (2) Larger Geo Eps | Silhouette: 0.271 | Davies-Bouldin: 1.092 | Calinski-Harabasz: 11993.433 | Noise Ratio: 0.322
Experiment: (2) Larger Time Eps | Silhouette: 0.452 | Davies-Bouldin: 0.742 | Calinski-Harabasz: 21261.104 | Noise Ratio: 0.535
Experiment: (2) Much Larger Time Eps | Silhouette: 0.474 | Davies-Bouldin: 0.677 | Calinski-Harabasz: 25

In [4]:
# Store all results.
exports_path = './data/exports/clusters/mv_experiments/evaluation_metrics/'
file_name = f"clustering_metrics_run_2"
store_pickle_variable(experiment_metrics, exports_path, file_name)

#### Tests

In [16]:
# print(df_experiment_features.iloc[-6:, [0, 2, 3, 7, 11, 15, 19, 23, 27, 31]].to_string(index=False))
print(df_experiment_features.iloc[10:11, :].to_string(index=False))

experiment_name                                                              file_name  num_clusters  time_length_mean  time_length_std  time_length_min  time_length_max  lat_length_mean  lat_length_std  lat_length_min  lat_length_max  lon_length_mean  lon_length_std  lon_length_min  lon_length_max  cluster_size_mean  cluster_size_std  cluster_size_min  cluster_size_max  cluster_volume_mean  cluster_volume_std  cluster_volume_min  cluster_volume_max  cluster_compactness_mean  cluster_compactness_std  cluster_compactness_min  cluster_compactness_max  cluster_eccentricity_mean  cluster_eccentricity_std  cluster_eccentricity_min  cluster_eccentricity_max  cluster_density_mean  cluster_density_std  cluster_density_min  cluster_density_max
Smaller Geo Eps clustering_experiment_2002-08-01_2024-07-01_sst_2_1_10_10_500_1000.pkl            27          2.555556          0.87489              2.0              5.0        46.185185       29.714739            16.0           172.0        50.592593    

In [21]:
single_exp_features = experiment_features[10]['features']
single_exp_features_df = pd.DataFrame(single_exp_features).T
print(single_exp_features_df.to_string())

    time_length  lat_length  lon_length  cluster_size  cluster_volume  cluster_compactness  cluster_eccentricity  cluster_density
1           2.0        16.0        19.0          69.0           608.0             0.113487              0.894737         0.113487
2           4.0        56.0        44.0        2889.0          9856.0             0.293121              0.928571         0.293121
4           5.0       172.0       153.0       27500.0        131580.0             0.208998              0.970930         0.208998
5           5.0        78.0       183.0       14254.0         71370.0             0.199720              0.972678         0.199720
6           3.0        32.0        23.0         543.0          2208.0             0.245924              0.906250         0.245924
7           3.0        49.0        40.0        1640.0          5880.0             0.278912              0.938776         0.278912
8           2.0        29.0        33.0         920.0          1914.0             0.480669